In [ ]:
import json
import numpy as np
import pandas as pd
from scipy.stats import pearsonr, spearmanr
from glob import glob
import os
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import Javascript, display
from IPython.display import HTML, Image
from pathlib import Path
import kaleido

In [ ]:
def read_results(path):
    #print(f"\tTrying to read {path}")
    if os.path.exists(path):
        with open(path) as f:
            results = json.load(f)
        return results
    else:
        print(f"Cannot find file {path}")
        return None

lang_map = {"eng": "en",
            "fra": "fr",
            "fin": "fi",
            "zho": "zh",
            "ita": "it",
            "deu": "de",
            "vie": "vi",
            "jpn": "ja",
            "ukr": "uk",
            "ell": "el",
            "hin": "hi",
            "bul": "bg",
            "fas": "fa"}

models = [os.path.basename(m) for m in glob("path_to_models")] + ["openai__text-embedding-3-small","openai__text-embedding-3-large", "gemini-embedding-001"]


dataset_map ={"ARCChallenge" : "arcchallenge",
              "SummEvalSummarization.v2": "summeval-2",
              "RTE3-multi": "rte3-multi__contradiction",
              "Tatoeba": "tatoeba",
              "WebFAQRetrieval": "web-faq-bitext"}

In [ ]:
def parse_mteb_results(models, dataset):
    if dataset == "RTE3-multi":
        return parse_rte(models, dataset)
    print("Reading MTEB results...")
    # path where the results are
    mteb_results_path=lambda x: [i for i in glob(f"mteb_out/{x}/results/{x}/*/{dataset}.json")]
    results = {}
    for model in models:
        #print(f"In {model}")
        if model == "gemini-embedding-001":
            model_to_read = "google__gemini-embedding-001"
        else:
            model_to_read = model
        mteb_results_path_ = mteb_results_path(model_to_read)
        if mteb_results_path_ == []:
            print(f"\tModel {model} missing results from mteb")
            continue
        if len(mteb_results_path_) != 1:
            # the dataset revision has changed, using the first results
            print(f"\t{len(mteb_results_path_)} evaluation files found for {model}. Selecting first results.")
            mteb_results_path_ = [mteb_results_path_[0]]
        
        d = read_results(mteb_results_path_[0])  # [0] just to remove nesting
        for subsplit in d["scores"]["test"]:
            #print(subsplit)
            if not f'{subsplit["hf_subset"]}' in results.keys():
                results[f'{subsplit["hf_subset"]}'] = {}
            results[f'{subsplit["hf_subset"]}'][f'{model}']= subsplit["main_score"]
    return results

def parse_rte(models, dataset):
    results = {}
    langs_rte = ["en", "fr", "it", "de"]
    mteb_results_path="mteb_out/rte_crawled.json"
    j = read_results(mteb_results_path)
    for m in models:
        m_ = m.replace("__", "/")
        if "gemini" in m_:
            m_ = "google/gemini-embedding-001"
        for l in langs_rte:
            if l not in results.keys():
                results[l] = {}
            #print(f"{m_}_{l}")
            results[l][m] = j[f"{m_}_{l}"]
    return results

In [ ]:
def parse_bss_results(models, dataset, splits, prompt=False, shuffled=False, method="ICA", dim=32):
    print("Reading BSS results...")
    ginis = {}
    peaks = {}
    peak_indices = {}
     # unneccessarily complex path
    bss_results_path=lambda x: f"stats{'_with_prompt' if prompt else ''}/{method}_{dim}/{dataset}/{x}.json" if isinstance(x, str) else f"stats{'_with_prompt' if prompt else ''}/{method}_{dim}/{dataset}:{x[1]}/{x[0]}.json"
   

    for m in models:
        if "openai" in m:
            m_ = m.replace("openai__","")
        else:
            m_ = m

        for s in splits:
            if s == "default":
                path = bss_results_path(m_)
            else:
                if dataset == "tatoeba":
                    # remap splits to match hf format
                    s_ = lang_map[s.split("-")[1]]+"-"+lang_map[s.split("-")[0]]
                    path = bss_results_path([m_, s_])
                else:
                    path = bss_results_path([m_, s])
            if not os.path.isfile(path):
                print(f"\tmissing results for {path}")
                continue
            else:
                j = read_results(path)
            if s not in ginis.keys():
                ginis[s] = {}
                peaks[s] = {}
                peak_indices[s] = {}
            ginis[s][m] = (dim-1)/dim - j[f"{'shuf_' if shuffled else ''}gini"]  # see analysis/README.md
            peaks[s][m] = j[f"{'shuf_' if shuffled else ''}peak"]
            peak_indices[s][m] = j["peak_indices"]
        
    return ginis, peaks, peak_indices




In [ ]:
dataset="ARCChallenge"
prompt = False
dataset_ = dataset_map[dataset]
mteb_scores = parse_mteb_results(models, dataset)
ginis32, peaks32, peak_indices32 = parse_bss_results(models, dataset_, splits = [i for i in mteb_scores.keys()], method="ICA", dim=32, shuffled=False, prompt=prompt)
ginis32s, peaks32s, peak_indices32s = parse_bss_results(models, dataset_, splits = [i for i in mteb_scores.keys()], method="ICA", dim=32, shuffled=True, prompt=prompt)
assert mteb_scores.keys() == ginis32.keys()

In [ ]:
def color_text(s):
    return f"\x1b[31m{s}\x1b[0m"

def analysis(dataset, df, score, against,split="default", pprint=False):
    collect_results = []
    if isinstance(score, str):
        score = [score]
    for s in score:
        x = df[s].tolist()
        y = df[against].tolist()
        pr = pearsonr(x,y)
        sp = spearmanr(x,y)
        p_value = round(pr.pvalue,5)
        p_value = color_text(p_value) if p_value <= 0.05 else p_value
        sp_value = round(sp.pvalue, 5)
        sp_value = color_text(sp_value) if sp_value <= 0.05 else sp_value
        if pprint:
            print(f"\t Statistic: {s} (support {len(x)})\t\tP-Result: {round(pr.statistic, 3)} (p-value {p_value})  \n\
                \t\t\t\tS-Result: {round(sp.statistic, 3)} (p-value {sp_value})")
        collect_results.append({"dataset": dataset, "split":split, "statistic":s, "support": len(x), "pr": (pr.statistic, pr.pvalue), "sp":(sp.statistic, sp.pvalue)})
    return collect_results
        

def make_table(dataset, mteb_scores, dict_of_results):
    results_bss = {}
    dataframes = {}
    for split in mteb_scores.keys():
        print(split)
        dfs = []
        df_mteb = pd.DataFrame([mteb_scores[split]]).T.rename(columns={0:"MTEB-score"})
        for name, result in dict_of_results.items():
            df = pd.DataFrame([result[split]]).T.rename(columns={0:name})
            dfs.append(df)
        df = pd.concat([df_mteb] + dfs,  axis=1)
        df = df.reset_index().rename(columns={"index":"model"})
        #display(df)
        df = df.dropna()   # drops all NaN values --> i.e. do not analyse unless all results exist
        r2 = analysis(dataset, df, [i for i in dict_of_results.keys() if "gini" in i or "peak" in i], "MTEB-score", split=split, pprint=True)
        results_bss[split]=r2
        dataframes[split] = df
    return results_bss, dataframes

results, dfs = make_table(dataset, mteb_scores, {"ginis32": ginis32, "ginis32_shuffled": ginis32s})
#make_table(dataset, mteb_scores, {"ginis32_shuffled": ginis32s})

In [ ]:
print(results)

In [ ]:
def plot_plotly_two_stats(df, title="Plot", return_fig=False):
    #max_label = max([int(i) for i in np.unique(df["num_peaks"])])
    #f max_label == 1:
    #    max_label = 2
    #palette = make_label_colormap(max_label=max_label)

    # Add instance ID to track which points belong together
    df = df.copy()
    df["instance_id"] = range(len(df))

    # Reshape: one row per (instance, statistic)
    df_long = df.melt(
        id_vars=["instance_id", "MTEB-score", "model"],
        value_vars=["ginis32", "ginis32_shuffled"],
        var_name="statistic_name",
        value_name="statistic_value"
    )

    fig = px.scatter(
        df_long,
        x="statistic_value",
        y="MTEB-score",
        color="statistic_name",
        symbol="statistic_name",          # different marker shape per statistic
        hover_data=["model", "statistic_name"],
        #custom_data=["url"],
        #color_discrete_sequence=palette,
        #category_orders={"num_peaks": [str(i) for i in range(0, max_label + 1)]},
        template="none",
        color_discrete_sequence=["red", "purple"],
        title=""
    )

    # Add lines connecting the two points of the same instance
    for instance_id, group in df_long.groupby("instance_id"):
        group_sorted = group.sort_values("statistic_name")  # consistent order
        fig.add_trace(
            go.Scatter(
                x=group_sorted["statistic_value"],
                y=group_sorted["MTEB-score"],
                mode="lines",
                line=dict(color="grey", width=1),
                opacity=0.4,
                showlegend=False,
                hoverinfo="skip",
            )
        )

    # Move lines behind the scatter points
    # Reorder traces: lines first, scatter on top
    n_scatter_traces = len(fig.data) - len(df_long["instance_id"].unique())
    reordered = list(fig.data[n_scatter_traces:]) + list(fig.data[:n_scatter_traces])
    fig.data = reordered

    fig.update_traces(
        selector=dict(mode="markers"),
        marker=dict(size=12)
    )
    fig.update_xaxes(title="Gini-index")
    fig.update_layout(width=1000, height=700, title=title, showlegend=False)


    if return_fig:
        return fig
    fig.show()
    #fig.write_image("shuffled_plot_bul_ICA32.svg")

for key in dfs.keys():
    print(key)
    print("unshuffled", results[key][0]["sp"], "shuffled", results[key][1]["sp"])
    plot_plotly_two_stats(dfs[key], title=f"{dataset} ({'paired' if prompt==False else 'shuffled'})")
